# AgentWatch model pipeline (Colab)

Runs the AgentWatch attribution model pipeline end to end on a fresh Colab machine, with the
repository's own commands:

1. clone the repository and pull dataset v1 from the public DVC remote (no login);
2. rebuild the frozen `split-v3` split and check it is byte-identical to the evaluated one;
3. optionally re-run the hyperparameter search on the validation split (about 10 to 20 minutes);
4. train the LightGBM classifier and regressor, score them on the test split, and calibrate;
5. draw the confusion matrices, model comparison, SHAP importance and tuning sensitivity.

Everything runs on CPU and stays on this machine: nothing is uploaded, and MLflow tracks the runs
in a local SQLite file. The settings, gates and thresholds are the repository's, declared before
evaluation; this notebook changes none of them. See `docs/model-training.md` for what each result
means and its limitations.

In [ ]:
# Which branch to run: "main" once the model PRs (#81, #82) are merged; until then their branch.
BRANCH = "main"
REPO = "https://github.com/all3n2601/AgentWatch.git"
RUN_TUNING = False  # True re-runs the 162-point validation search (slow)

In [ ]:
import os
import subprocess


def sh(command: str) -> None:
    """Run a shell command from the repository and stop the notebook if it fails."""
    print(f"$ {command}", flush=True)
    subprocess.run(command, shell=True, check=True, cwd="/content/AgentWatch")


if not os.path.isdir("/content/AgentWatch"):
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, "/content/AgentWatch"],
        check=True,
    )
os.chdir("/content/AgentWatch")
sh("git log --oneline -1")

## Environment

`uv` installs the pinned Python (3.13) and the locked dependencies (`uv.lock`), so the run uses
the same package versions as the repository's CI. LightGBM needs OpenMP, which Colab's Linux
image provides.

In [ ]:
sh("pip -q install uv==0.13.8")  # the uv the lockfile was made with
sh("uv --version")
sh("uv sync --frozen --all-packages --quiet")

## Data

Dataset v1 (`steps-v1.3+aafae73eb211`) and its evidence come from the public DVC remote. The
`split-v3` file is rebuilt from the release by the frozen split code and compared with the SHA-256
recorded for the evaluated split.

In [ ]:
SPLIT = "data/datasets/splits/steps-v1.3+aafae73eb211/split-v3+932b2df410a7.json"
sh("make pull-dataset")
sh(
    "uv run --frozen --package agentwatch-data python -m agentwatch_data.splits "
    "--release data/datasets/steps-v1.3+aafae73eb211"
)

In [ ]:
import hashlib

EXPECTED_SPLIT_SHA256 = "32488a942ad66f567be84e5992c2fe0bb14c92b6f67a4238000b85644f06d7c3"
from pathlib import Path

actual = hashlib.sha256(Path(SPLIT).read_bytes()).hexdigest()
assert actual == EXPECTED_SPLIT_SHA256, f"split differs from the evaluated one: {actual}"
print("split-v3 matches the evaluated split")

## Tuning (optional)

`make tune-candidates` searches the grid declared in `pipelines/model/src/agentwatch_model/tuning.toml`
on the validation split only; it never reads the test split. Its result is already recorded in the
packaged `training.toml`, so skipping this cell changes no result.

In [ ]:
if RUN_TUNING:
    sh(f"make tune-candidates SPLIT={SPLIT}")
else:
    print("Skipped; set RUN_TUNING = True to re-run the validation search.")

## Train, evaluate and calibrate

`make train-candidates` trains on the training split (early stopping on validation) with the
packaged `training.toml`, scores the model and the `rules-v1` baseline on the same test rows
against the evaluation contract, and saves the models. `make calibrate` fits temperature scaling
and the abstention threshold on validation and scores them on test.

In [ ]:
sh(f"make train-candidates SPLIT={SPLIT}")
sh(f"make calibrate SPLIT={SPLIT}")

In [ ]:
import json
from pathlib import Path

candidates = json.loads(Path(".data/candidates/report.json").read_text())
calibration = json.loads(Path(".data/calibration/report.json").read_text())
gates = candidates["gates"]
print("model         ", candidates["model"]["version"], candidates["status"])
print(
    "macro F1      ",
    round(candidates["comparison"]["macro_f1_all_classes"]["model"], 3),
    "(rules-v1",
    round(candidates["comparison"]["macro_f1_all_classes"]["rules"], 3),
    ")",
)
print(
    "clean -> none ",
    round(gates["clean_none_rate"]["value"], 3),
    gates["clean_none_rate"]["outcome"],
)
print(
    "MAE           ",
    round(gates["regression_error"]["value"], 3),
    gates["regression_error"]["outcome"],
)
print(
    "calibration   ",
    calibration["calibration"]["version"],
    "T",
    round(calibration["calibration"]["temperature"], 3),
    "threshold",
    round(calibration["calibration"]["abstention_threshold"], 3),
)
print("gate outcomes ", candidates["summary"]["outcomes"])

## Figures

`make model-analysis` reads the reports and the saved classifier, after checking they belong to
this split; it trains and selects nothing. SHAP values are LightGBM's exact TreeSHAP
contributions on the test rows.

In [ ]:
from IPython.display import Image, display

sh(f"make model-analysis SPLIT={SPLIT}")
figures = ["comparison", "confusion-rules", "confusion-selected", "shap-selected"]
if RUN_TUNING:
    figures.append("sensitivity")
for name in figures:
    path = f".data/analysis/{name}.png"
    assert os.path.exists(path), f"{path} was not written"
    print(name)
    display(Image(path))

## Reading the results

- Settings were chosen on the validation split, which is 273 steps from 3 experiments of one task;
  the test split is one unseen task, so there is no interval and no causal claim.
- `docs/model-training.md` records how each training version was chosen and how it compares with
  the earlier ones on test; the repository reports test results rather than choosing on them.
- Four contract gates are `not_evaluated`, so the model is not promotable yet.